# EPIC Clarity Observation Period Hydration

This notebook hydrates the OMOP OBSERVATION_PERIOD table from EPIC Clarity event dates.

## Source Tables
- `_exponent._bronze_epic_clarity_*.dbo_PAT_ENC` - Patient encounters (observation start)
- `_exponent._bronze_epic_clarity_*.dbo_ORDER_MED` - Medication orders (observation end)
- `_exponent._bronze_epic_clarity_*.dbo_ORDER_PROC` - Procedures (observation end)
- `_exponent._bronze_epic_clarity_*.dbo_ORDER_RESULTS` - Lab results (observation end)

## OMOP Fields Populated
- person_id
- observation_period_start_date (MIN encounter date)
- observation_period_end_date (MAX of all event dates)
- period_type_concept_id (32809 = EHR administrative record)

## Notes
- Aggregates dates across multiple event types
- Start date is earliest encounter date per patient
- End date is latest of any clinical event date

In [ ]:
source = 'epic_clarity'

In [ ]:
-- Silver Layer: Determine observation periods from encounter and event data
%sql
CREATE OR REPLACE TEMP VIEW observation_period_silver AS
SELECT
    CONCAT_WS(CHR(31), 'epic_clarity', 'PATIENT', 'PAT_ID', e.PAT_ID) AS person_source_value,
    MIN(e.observation_start_date) AS observation_period_start_date,
    MAX(e.observation_end_date) AS observation_period_end_date,
    CURRENT_TIMESTAMP() AS updated_tsp
FROM (
    -- Encounter-based observation periods
    SELECT PAT_ID, CONTACT_DATE AS observation_start_date, NULL AS observation_end_date
    FROM _exponent._bronze_epic_clarity_prod01_vw.dbo_PAT_ENC
    UNION ALL
    -- Medication order dates
    SELECT p.PAT_ID, NULL AS observation_start_date, om.ORDERING_DATE AS observation_end_date
    FROM _exponent._bronze_epic_clarity_prod01_vw.dbo_ORDER_MED om
    JOIN _exponent._bronze_epic_clarity_prod01_vw.dbo_PAT_ENC p
        ON om.PAT_ENC_CSN_ID = p.PAT_ENC_CSN_ID
    UNION ALL
    -- Procedure order dates
    SELECT p.PAT_ID, NULL, op.ORDERING_DATE
    FROM _exponent._bronze_epic_clarity_prod01_vw.dbo_ORDER_PROC op
    JOIN _exponent._bronze_epic_clarity_prod01_vw.dbo_PAT_ENC p
        ON op.PAT_ENC_CSN_ID = p.PAT_ENC_CSN_ID
    UNION ALL
    -- Result/lab dates
    SELECT p.PAT_ID, NULL, ore.RESULT_DATE
    FROM _exponent._bronze_epic_clarity_prod01_vw.dbo_ORDER_RESULTS ore
    JOIN _exponent._bronze_epic_clarity_prod01_vw.dbo_PAT_ENC p
        ON ore.PAT_ENC_CSN_ID = p.PAT_ENC_CSN_ID
) e
WHERE e.PAT_ID IS NOT NULL
GROUP BY e.PAT_ID

In [ ]:
-- Merge into Silver Layer
%sql
MERGE INTO _exponent.omop_silver.observation_period AS target
USING observation_period_silver AS source
ON target.person_source_value = source.person_source_value

WHEN MATCHED AND NOT (
    target.observation_period_start_date <=> source.observation_period_start_date
    AND target.observation_period_end_date <=> source.observation_period_end_date
)
THEN UPDATE SET
    target.observation_period_start_date = source.observation_period_start_date,
    target.observation_period_end_date = source.observation_period_end_date,
    target.updated_tsp = source.updated_tsp

WHEN NOT MATCHED THEN INSERT (
    person_source_value,
    observation_period_start_date,
    observation_period_end_date,
    updated_tsp
)
VALUES (
    source.person_source_value,
    source.observation_period_start_date,
    source.observation_period_end_date,
    source.updated_tsp
)

In [ ]:
-- Populate mapping table
%sql
INSERT INTO _exponent.omop_mapping.source_to_observation_period (
    source_system,
    person_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    'epic_clarity' AS source_system,
    s.person_source_value,
    TRUE AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    s.updated_tsp AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT person_source_value, updated_tsp
    FROM _exponent.omop_silver.observation_period
    WHERE person_source_value IS NOT NULL
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_observation_period x
    ON s.person_source_value = x.person_source_value
    AND x.source_system = 'epic_clarity'

In [ ]:
-- Gold Layer: Join with person mapping to get person_id
%sql
CREATE OR REPLACE TEMP VIEW observation_period_gold AS
SELECT
    m_person.person_id,
    s.observation_period_start_date,
    s.observation_period_end_date,
    s.updated_tsp
FROM _exponent.omop_silver.observation_period s
INNER JOIN _exponent.omop_mapping.source_to_person m_person
    ON s.person_source_value = m_person.person_source_value
    AND m_person.source_system = 'epic_clarity'
    AND m_person.active_flag = TRUE

In [ ]:
-- Merge into Gold Layer (OMOP)
%sql
MERGE INTO _exponent.omop.observation_period AS target
USING observation_period_gold AS source
ON target.person_id = source.person_id
    AND target.observation_period_start_date = source.observation_period_start_date

WHEN MATCHED AND NOT (
    target.observation_period_end_date <=> source.observation_period_end_date
)
THEN UPDATE SET
    target.observation_period_end_date = source.observation_period_end_date

WHEN NOT MATCHED THEN INSERT (
    person_id,
    observation_period_start_date,
    observation_period_end_date,
    period_type_concept_id
)
VALUES (
    source.person_id,
    source.observation_period_start_date,
    source.observation_period_end_date,
    32809  -- EHR administrative record
)